<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1: Introducing Polars

This notebook accompanies Chapter 1 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**. The book was written against Polars 1.20, so a few details differ from the printed text. They are called out where they matter.

Polars is a fast DataFrame library written in Rust. In this notebook you will run a complete **ETL** (extract, transform, load) workflow on a real dataset: every Citi Bike trip taken in New York City in March 2024 (about 2.6 million rows).

**Don't worry about understanding every line yet.** The goal of this chapter is to show you what Polars code looks like. Later chapters explain each concept in depth. The explanations in this notebook focus on *why* each step is written the way it is.

By the end of this notebook you will have seen how to:

- read large CSV files efficiently with Polars' lazy API (`scan_csv`)
- clean data with reusable **expressions**
- compute new columns, such as trip distance
- enrich a DataFrame by **joining** it with other data, safely
- aggregate data over time with `group_by_dynamic`
- write the result to partitioned **Parquet** files
- build the same pipeline lazily, and **prove** it gives the same answer as the step-by-step version

**How to use this notebook:** run the cells from top to bottom. Cells depend on earlier cells, so if something goes wrong, choose *Runtime → Restart session* and *Run all*.

## 0. Setup

### Install the packages

Colab comes with some packages preinstalled, but not necessarily the versions we want. We install:

| Package | What we use it for |
|---|---|
| `polars` | the DataFrame library itself, **pinned to the 2.1 series** |
| `plotnine` | plotting with a "grammar of graphics" API |
| `mizani` | axis breaks and labels for plotnine |
| `shapely` | geometry operations (point-in-polygon tests) |

**Why pin the Polars version?** Polars evolves quickly, and APIs get renamed or removed between versions. Pinning means everyone running this notebook gets the same behavior as the author. `polars==2.1.*` means "any 2.1.x release".

We use `%pip` (not `!pip`) so the package is installed into the environment of the *running* notebook kernel.

In [ ]:
%pip install -q "polars==2.1.*" plotnine mizani shapely

### Import the packages and check the version

If Polars was already imported before the install above, Colab keeps using the old version until the session is restarted. The assertion below turns that silent problem into a clear error message.

We import plotnine as `p9` rather than with `from plotnine import *`. Star imports dump hundreds of names into your namespace, which makes it hard to tell where a function comes from.

In [ ]:
import math
from pathlib import Path

import plotnine as p9
import polars as pl
import shapely
from mizani.breaks import breaks_date
from mizani.labels import label_comma
from polars.testing import assert_frame_equal
from shapely import STRtree

assert pl.__version__.startswith("2.1"), (
    f"This notebook expects Polars 2.1.x but found {pl.__version__}. "
    "Re-run the install cell, then choose Runtime > Restart session and run all cells again."
)
print(f"Polars {pl.__version__}")

### Notebook-wide settings

Defining file paths and settings **once**, at the top, means we never have to repeat (and possibly mistype) them later.

- `SCHEMA_OVERRIDES` tells Polars the data types of four CSV columns. Polars infers types by looking at the first rows of a file. The two `*_station_id` columns look numeric at first but contain text later on, so we force them to `String`. The two timestamp columns are declared as `Datetime` so they are parsed into real timestamps, not text.
- `BIKE_TYPE` and `RIDER_TYPE` are **Enums**: a data type for text columns whose possible values are known in advance. They use little memory, remember the order of their categories, and raise an error if an unexpected value appears. That strictness is a feature: it catches data surprises early.
- `pl.Config.set_tbl_cols(-1)` makes Polars show *all* columns when it displays a table.

In [ ]:
DATA_DIR = Path("sample_data/citibike")
TRIPS_GLOB = str(DATA_DIR / "202403-citibike-tripdata_*.csv")
GEOJSON_PATH = DATA_DIR / "nyc-neighborhoods.geojson"

SCHEMA_OVERRIDES = {
    "started_at": pl.Datetime,
    "ended_at": pl.Datetime,
    "start_station_id": pl.String,
    "end_station_id": pl.String,
}

BIKE_TYPE = pl.Enum(["classic", "electric"])
RIDER_TYPE = pl.Enum(["member", "casual"])

pl.Config.set_tbl_cols(-1)

## An ETL Showcase

**ETL** stands for **E**xtract, **T**ransform, **L**oad:

1. **Extract**: get the raw data from its sources.
2. **Transform**: clean it and add useful information.
3. **Load**: write the result somewhere it can be reused.

We also add two data-visualization "bonuses" along the way.

### Extract

We use two sources: the **bike trips** (Citi Bike's public data) and a **GeoJSON** file describing New York City's neighborhoods and boroughs.

#### Download the Citi Bike trips

These cells run shell commands. In a notebook, `%%bash` at the top of a cell runs the whole cell as a shell script. The download is skipped if the files are already there, so re-running the cell is harmless. The flags `-fsSL` make `curl` fail loudly on HTTP errors (instead of saving an error page as if it were data), follow redirects, and stay quiet otherwise.

Note that Colab's storage is **temporary**: files disappear when the runtime is recycled, so you may need to download again in a new session.

In [ ]:
%%bash

curl -sO https://s3.amazonaws.com/tripdata/202403-citibike-tripdata.zip
unzip -o 202403-citibike-tripdata.zip -d sample_data/citibike/
rm -f 202403-citibike-tripdata.zip

In [ ]:
%%bash
set -euo pipefail
mkdir -p sample_data/citibike

curl -fsSL -o sample_data/citibike/trips.zip https://s3.amazonaws.com/tripdata/202403-citibike-tripdata.zip
unzip -o -j -q sample_data/citibike/trips.zip "*.csv" -x -d sample_data/citibike/
rm -f sample_data/citibike/trips.zip

ls -lh sample_data/citibike

#### Download the neighborhood boundaries

The GeoJSON file describes each neighborhood as a **polygon**, a sequence of longitude/latitude coordinates outlining its borders.

In [ ]:
%%bash
set -euo pipefail

curl -fsSL -o sample_data/citibike/nyc-neighborhoods.geojson \
    https://raw.githubusercontent.com/davidrpugh/data-analytics-with-polars/refs/heads/main/data/citibike/nyc-neighborhoods.geojson
ls -lh sample_data/citibike/nyc-neighborhoods.geojson

#### Inspect the raw files before loading them

A good habit: look at raw data *before* parsing it. Here we count the lines in each CSV file (`wc -l`, one line per trip plus a header) and peek at the first few rows of one file (`head`). We're checking that the files have a header, use commas as separators, and look like what we expect.

In [ ]:
! wc -l sample_data/citibike/202403-citibike-tripdata_*.csv

In [ ]:
! head -n 6 sample_data/citibike/202403-citibike-tripdata_1.csv

#### Look before you cast

Our Enum types only allow specific values. Before relying on that, let's check which distinct values actually occur in the two categorical columns. `scan_csv` (explained next) lets us do this without loading the entire dataset into memory, and `implode()` collects each column's distinct values into a single list so they print compactly.

**If you see a value other than the ones in `BIKE_TYPE` / `RIDER_TYPE`** (for example `docked_bike`), add it to the Enum definition in the settings cell. Otherwise the cast below will raise an error, which is exactly the point of using Enums.

In [ ]:
pl.scan_csv(TRIPS_GLOB, schema_overrides=SCHEMA_OVERRIDES).select(
    rideable_type=pl.col("rideable_type").unique().sort().implode(),
    member_casual=pl.col("member_casual").unique().sort().implode(),
).collect()

#### Read the trips with the lazy API

Polars has two ways to read a CSV file:

- `pl.read_csv(...)` is **eager**: it immediately loads *everything* into a `DataFrame`.
- `pl.scan_csv(...)` is **lazy**: it returns a `LazyFrame`, which is only a *plan*. Nothing is read until you call `.collect()`.

Laziness lets Polars optimize the plan first. In particular, if we `select` only some columns, Polars reads *only those columns* from disk. The raw files have 13 columns, including `ride_id` and two station-ID columns we don't need; those are never even parsed.

The function below describes the work: scan all the CSV files (the `*` in `TRIPS_GLOB` matches every part), keep and rename the columns we need, convert the two categorical columns to Enums, and compute each trip's `duration`.

We wrap it in a **function** so we can reuse the *exact same code* in the lazy-pipeline bonus at the end of the notebook. Defining it once means the two versions can't accidentally drift apart.

Notes on the expressions:
- `pl.col("rideable_type").str.split("_").list.first()` turns `"classic_bike"` into `"classic"`.
- Subtracting one Datetime column from another gives a `Duration` column.

In [ ]:
def scan_trips() -> pl.LazyFrame:
    """Lazily scan the Citi Bike CSV files and return the columns we need, with tidy names."""
    return (
        pl.scan_csv(TRIPS_GLOB, schema_overrides=SCHEMA_OVERRIDES)
        .select(
            bike_type=pl.col("rideable_type").str.split("_").list.first().cast(BIKE_TYPE),
            rider_type=pl.col("member_casual").cast(RIDER_TYPE),
            datetime_start=pl.col("started_at"),
            datetime_end=pl.col("ended_at"),
            station_start=pl.col("start_station_name"),
            station_end=pl.col("end_station_name"),
            lon_start=pl.col("start_lng"),
            lat_start=pl.col("start_lat"),
            lon_end=pl.col("end_lng"),
            lat_end=pl.col("end_lat"),
        )
        .with_columns(duration=pl.col("datetime_end") - pl.col("datetime_start"))
    )

Now we call `.collect()` to execute the plan and materialize a `DataFrame`. Look at the header of the table below: Polars shows each column's **name** and **data type**, as well as the table's **shape** (rows × columns). pandas doesn't show data types this way.

In [ ]:
trips_raw = scan_trips().collect()
trips_raw

#### Read in the neighborhoods from GeoJSON

First, a peek at the raw file. GeoJSON is deeply nested JSON, which is not a rectangular table, so we can't use it as a DataFrame directly. We only print the first 1,500 characters because the full file is very long.

In [ ]:
! head -c 1500 data/citibike/nyc-neighborhoods.geojson

We turn this nested structure into a rectangular DataFrame with a chain of methods. Read the chain from top to bottom:

1. `read_json`: load the file. The result has a single `features` column holding one big list.
2. `explode("features")`: turn each list element into its own **row**.
3. `unnest("features")`, `unnest("properties")`, `unnest("geometry")`: turn the keys of nested **structs** into their own **columns**.
4. `list.first()` picks the outer ring of coordinates, which is the neighborhood's boundary. We call this `polygon`.
5. We drop Staten Island (it has no Citi Bike stations) and sort by name.

If a neighborhood consists of several separate areas, it appears once per area. You'll learn about `explode` and `unnest` in Chapter 15.

In [ ]:
neighborhoods = (
    pl.read_json(GEOJSON_PATH)
    .select("features")
    .explode("features")
    .unnest("features")
    .unnest("properties")
    .select("neighborhood", "borough", "geometry")
    .unnest("geometry")
    .with_columns(polygon=pl.col("coordinates").list.first())
    .select("neighborhood", "borough", "polygon")
    .filter(pl.col("borough") != "Staten Island")
    .sort("neighborhood")
)

neighborhoods

### Transform

No dataset is perfect, and this one isn't either. In this step we clean the rows, add each trip's distance, and add the borough and neighborhood where each trip started and ended.

#### Reusable building blocks: expressions

An **expression** is a *description* of a computation, such as "the trip's duration is less than five minutes". By itself an expression does nothing. It only runs when you pass it to a method such as `filter` or `with_columns`. Because expressions are ordinary Python objects, you can **name them and reuse them**, in an eager pipeline or a lazy one. We use that below to define each cleaning rule exactly once.

- `in_march`: the trip started and ended in March 2024. A few trips started on Feb 29 and ended in March, and we want to remove them.
- `is_not_a_trip`: the rider returned the bike to the **same station** in **under five minutes**, which means they probably never rode anywhere (e.g., a docking problem). Comparing the `duration` column directly with `pl.duration(minutes=5)` is clearer than converting to seconds first.
- `haversine_distance`: the distance between two points on a sphere. This is "as the crow flies", not the distance the bike actually traveled. The book computes this with a custom Rust plugin; here we express the formula directly with Polars expressions, which keeps the notebook self-contained and still runs fast. The formula is: `a = sin²(Δlat/2) + cos(lat1)·cos(lat2)·sin²(Δlon/2)`, then `distance = 2·R·arcsin(√a)`, where `R` is the Earth's radius.
- `REQUIRED_COLUMNS`: the columns that must not be null. We list them explicitly (rather than calling `drop_nulls()` on everything) so it's clear which missing values we care about.

In [ ]:
EARTH_RADIUS_KM = 6371.0088  # mean Earth radius

REQUIRED_COLUMNS = [
    "datetime_start", "datetime_end",
    "station_start", "station_end",
    "lon_start", "lat_start", "lon_end", "lat_end",
]

FINAL_COLUMNS = [
    "bike_type", "rider_type",
    "datetime_start", "datetime_end", "duration",
    "station_start", "station_end",
    "neighborhood_start", "neighborhood_end",
    "borough_start", "borough_end",
    "lat_start", "lon_start", "lat_end", "lon_end",
    "distance",
]

in_march = (pl.col("datetime_start") >= pl.datetime(2024, 3, 1)) & (
    pl.col("datetime_end") < pl.datetime(2024, 4, 1)
)

is_not_a_trip = (pl.col("station_start") == pl.col("station_end")) & (
    pl.col("duration") < pl.duration(minutes=5)
)


def haversine_distance(lon1: pl.Expr, lat1: pl.Expr, lon2: pl.Expr, lat2: pl.Expr) -> pl.Expr:
    """Great-circle distance in kilometers between two (longitude, latitude) points in degrees."""
    lon1, lat1, lon2, lat2 = (angle.radians() for angle in (lon1, lat1, lon2, lat2))
    a = ((lat2 - lat1) / 2).sin() ** 2 + lat1.cos() * lat2.cos() * ((lon2 - lon1) / 2).sin() ** 2
    return 2 * EARTH_RADIUS_KM * a.sqrt().arcsin()


distance_km = haversine_distance(
    pl.col("lon_start"), pl.col("lat_start"), pl.col("lon_end"), pl.col("lat_end")
)

**Test your expressions.** Because expressions can run on literal values, we can sanity-check `haversine_distance` without any trip data: one degree of longitude along the equator should be exactly `2·π·R / 360` kilometers (about 111.2 km). If the formula had a typo, this assertion would fail immediately.

In [ ]:
one_degree_km = pl.select(
    haversine_distance(pl.lit(0.0), pl.lit(0.0), pl.lit(1.0), pl.lit(0.0))
).item()

assert math.isclose(one_degree_km, 2 * math.pi * EARTH_RADIUS_KM / 360, rel_tol=1e-9)
one_degree_km

#### Clean up the rows

We apply the rules one at a time, keeping each intermediate DataFrame in its own variable. DataFrames in Polars are **immutable**: every operation returns a *new* DataFrame, and the old one is unchanged. That's why we can give each stage its own name, and why re-running a cell never corrupts earlier results.

The table at the end is a *cleaning log*. Whenever you remove data, **record how much you removed**, so you (and your readers) know the effect of each rule. `shift(1)` looks at the previous row, so `removed` is the previous row count minus the current one.

In [ ]:
trips_no_nulls = trips_raw.drop_nulls(subset=REQUIRED_COLUMNS)
trips_in_march = trips_no_nulls.filter(in_march)
trips_clean = trips_in_march.filter(~is_not_a_trip)

cleaning_log = pl.DataFrame(
    {
        "step": [
            "raw trips",
            "dropped rows with missing values",
            "kept only trips within March 2024",
            "dropped round trips under 5 minutes",
        ],
        "rows": [trips_raw.height, trips_no_nulls.height, trips_in_march.height, trips_clean.height],
    }
).with_columns(removed=pl.col("rows").shift(1) - pl.col("rows"))

cleaning_log

#### Add the trip distance

`with_columns` adds new columns to a DataFrame (or replaces existing ones with the same name) and keeps all the existing columns. We pass it the `distance_km` expression defined above.

In [ ]:
trips_with_distance = trips_clean.with_columns(distance=distance_km)

trips_with_distance.select(
    "lon_start", "lat_start", "lon_end", "lat_end", "distance", "duration"
)

#### Add the borough and neighborhood

The trips contain station *names* and coordinates, but not the neighborhood they are in. To find out, we first build a table with **one row per station**, then test which neighborhood polygon contains each station.

**Step 1: one row per station.** For each station we use the *median* start coordinates of all trips that began there. The median is robust to the occasional GPS glitch.

In [ ]:
def compute_stations(trips: pl.DataFrame) -> pl.DataFrame:
    """One row per start station, with the median start coordinates of its trips."""
    return (
        trips.group_by(station=pl.col("station_start"))
        .agg(
            lon=pl.col("lon_start").median(),
            lat=pl.col("lat_start").median(),
        )
        .sort("station")
    )


stations = compute_stations(trips_clean)
stations

**Step 2: which polygon contains each station?**

The book does this with a *cross join*: pair every station with every neighborhood (about 2,000 × 250 = 500,000 pairs) and run a point-in-polygon test on each pair. That is slow, and building a polygon object from scratch for every pair is slower still.

A better approach is a **spatial index**. We build each of the ~250 polygons **once**, put them in an `STRtree` (a tree structure that quickly narrows down which polygons *could* contain a point), and then ask the tree about all stations at once. Polars does what it's best at (tables and joins) and Shapely does what it's best at (geometry).

Three details worth noticing:

1. **`validate="1:m"` / `"m:1"`** on the joins states the relationship we *expect* between the two tables. If the data ever violates it (say, duplicate keys), Polars raises an error instead of silently producing a wrong result.
2. **Some stations lie on a border** and match more than one neighborhood. The book's `unique("station")` picks an arbitrary match. We sort by polygon index first and keep the first, so the result is **deterministic**. We also record `n_matches` so we can report how often this happens.
3. **Some stations match no polygon at all**, for example ones just outside the boundaries. They drop out of the result and we count them.

In [ ]:
def locate_stations(stations: pl.DataFrame, neighborhoods: pl.DataFrame) -> pl.DataFrame:
    """Find the borough and neighborhood containing each station (point-in-polygon)."""
    polygons = [shapely.Polygon(ring) for ring in neighborhoods["polygon"].to_list()]
    tree = STRtree(polygons)
    points = shapely.points(stations["lon"].to_numpy(), stations["lat"].to_numpy())

    # Pairs of (station index, polygon index) where the station is inside the polygon.
    station_idx, polygon_idx = tree.query(points, predicate="within")
    matches = pl.DataFrame(
        {"station_idx": station_idx, "polygon_idx": polygon_idx}
    ).with_columns(pl.all().cast(pl.UInt32))  # match the dtype of with_row_index()

    return (
        stations.with_row_index("station_idx")
        .join(matches, on="station_idx", validate="1:m")
        .join(
            neighborhoods.with_row_index("polygon_idx").select(
                "polygon_idx", "borough", "neighborhood"
            ),
            on="polygon_idx",
            validate="m:1",
        )
        .with_columns(n_matches=pl.len().over("station"))
        .sort("polygon_idx")
        .unique("station", keep="first", maintain_order=True)
        .select("station", "borough", "neighborhood", "n_matches")
        .sort("station")
    )

In [ ]:
located = locate_stations(stations, neighborhoods)
station_neighborhoods = located.drop("n_matches")

n_unmatched = stations.height - located.height
n_multi = located.filter(pl.col("n_matches") > 1).height
print(f"{stations.height:,} stations in total")
print(f"{n_unmatched:,} stations matched no neighborhood (dropped)")
print(f"{n_multi:,} stations matched more than one neighborhood (first match kept)")

station_neighborhoods

#### Bonus: visualize the neighborhoods and stations

Now that we know where the stations are, let's draw them. plotnine expects data in **long format**, one row per coordinate, so we reshape the polygons first: `explode` turns each polygon (a list of coordinates) into one row per coordinate, and then we split each `[lon, lat]` pair into two columns.

The plot is built by adding **layers** with `+`. Each layer says *what to draw* (`geom_*`) and which columns map to which visual property (`aes`). The subtitle is computed from the data with an f-string, so it can never fall out of sync with the numbers.

The shading of the neighborhoods only helps tell them apart; it has no meaning. The second layer (the stations) needs `inherit_aes=False` because it uses a different DataFrame that doesn't have the `id` column the first layer's `group` aesthetic refers to.

In [ ]:
neighborhoods_coords = (
    neighborhoods.with_row_index("id")
    .explode("polygon")
    .with_columns(
        lon=pl.col("polygon").list.first(),
        lat=pl.col("polygon").list.last(),
    )
    .drop("polygon")
)

stations_located = stations.join(station_neighborhoods, on="station", validate="1:1")
n_stations = stations_located.height
n_neighborhoods = stations_located["neighborhood"].n_unique()

(
    p9.ggplot(neighborhoods_coords, p9.aes(x="lon", y="lat", group="id"))
    + p9.geom_polygon(p9.aes(alpha="neighborhood", fill="borough"), color="white")
    + p9.geom_point(
        data=stations_located,
        mapping=p9.aes(x="lon", y="lat"),
        size=0.1,
        inherit_aes=False,
    )
    + p9.scale_x_continuous(expand=(0, 0))
    + p9.scale_y_continuous(expand=(0, 0, 0, 0.01))
    + p9.scale_alpha_ordinal(range=(0.3, 1))
    + p9.scale_fill_brewer(type="qual", palette=2)
    + p9.guides(alpha="none")
    + p9.labs(
        title="New York City neighborhoods and Citi Bike stations",
        subtitle=f"{n_stations:,} stations across {n_neighborhoods:,} neighborhoods",
        caption="Source: https://citibikenyc.com/system-data",
        fill="Borough",
    )
    + p9.theme_void(base_size=14)
    + p9.theme(
        dpi=150,
        figure_size=(7, 9),
        plot_background=p9.element_rect(fill="white", color="white"),
        plot_caption=p9.element_text(style="italic"),
        plot_margin=0.01,
        plot_title=p9.element_text(ha="left"),
    )
)

#### Join the neighborhoods back onto the trips

Now we attach the borough and neighborhood to every trip, **twice**: once for the station where the trip started and once for the station where it ended. Each join uses a copy of `station_neighborhoods` whose columns carry a suffix (`_start` or `_end`), which is what `pl.all().name.suffix(...)` does.

`validate="m:1"` says "many trips map to one station". That's true as long as station names are unique in the lookup table, and Polars will check it for us.

Two things to notice:

- A default (inner) join **silently drops** trips whose station isn't in the lookup table. We print how many rows were lost so it's never a surprise.
- The final `sort("datetime_start")` matters. Joins don't guarantee row order, and the `group_by_dynamic` aggregation below needs the time column to be sorted.

In [ ]:
def add_boroughs_and_neighborhoods(
    trips: pl.DataFrame, station_neighborhoods: pl.DataFrame
) -> pl.DataFrame:
    """Attach borough and neighborhood for both the start and the end station of each trip."""
    return (
        trips.join(
            station_neighborhoods.select(pl.all().name.suffix("_start")),
            on="station_start",
            validate="m:1",
        )
        .join(
            station_neighborhoods.select(pl.all().name.suffix("_end")),
            on="station_end",
            validate="m:1",
        )
        .select(FINAL_COLUMNS)
        .sort("datetime_start")
    )


trips_enriched = add_boroughs_and_neighborhoods(trips_with_distance, station_neighborhoods)

n_lost = trips_with_distance.height - trips_enriched.height
print(f"{n_lost:,} trips lost because a station could not be placed in a neighborhood")

trips_enriched

#### Bonus: visualize the daily trips per borough

With boroughs attached, we can aggregate. `group_by_dynamic` is a *time-aware* group-by: it cuts the time axis into fixed windows (here `every="1d"`, one day), optionally within groups (`group_by="borough_start"`), and `agg` summarizes each window. `pl.len()` counts the rows in each group.

(The book calls this variable `trips_per_hour`, but the windows are days, so we use a name that matches what it holds.)

In [ ]:
trips_per_day = (
    trips_enriched.sort("datetime_start")  # group_by_dynamic requires a sorted time column
    .group_by_dynamic("datetime_start", group_by="borough_start", every="1d")
    .agg(num_trips=pl.len())
)

trips_per_day

In [ ]:
(
    p9.ggplot(
        trips_per_day,
        p9.aes(x="datetime_start", y="num_trips", fill="borough_start"),
    )
    + p9.geom_area()
    + p9.scale_fill_brewer(type="qual", palette=2)
    + p9.scale_x_datetime(
        breaks=breaks_date(width="1 day"), date_labels="%-d", expand=(0, 0)
    )
    + p9.scale_y_continuous(labels=label_comma(), expand=(0, 0))
    + p9.labs(
        x="March 2024",
        fill="Borough",
        y="Trips per day",
        title="Citi Bike trips per day in March 2024",
        subtitle="On March 23, nearly 10 cm of rain fell in NYC",
    )
    + p9.theme_tufte(base_size=14)
    + p9.theme(
        axis_ticks_major=p9.element_line(color="white"),
        figure_size=(8, 5),
        legend_position="top",
        plot_background=p9.element_rect(fill="white", color="white"),
        plot_caption=p9.element_text(style="italic"),
        plot_title=p9.element_text(ha="left"),
    )
)

### Load

The last step of an ETL workflow is to **load** the result: write it somewhere it can be reused. We write **Parquet** files instead of CSV. Compared with CSV, Parquet:

- stores the **data type** of each column (the *schema*), so nothing has to be guessed when reading it back
- is **columnar**, which makes reading a few columns much faster
- keeps **statistics** per chunk of data, which lets readers skip chunks they don't need
- is **compressed**, so files are much smaller

We write **one file per day**. `partition_by` splits a DataFrame into several DataFrames, one per unique value of a column. With `as_dict=True` we get a dictionary whose keys are tuples (even for a single column), and `include_key=False` leaves the `date` column out of the files because the file name already says which day it is.

In [ ]:
trips_by_date = trips_enriched.with_columns(
    date=pl.col("datetime_start").dt.date().cast(pl.String)
).partition_by("date", as_dict=True, include_key=False)

for (date,), df in trips_by_date.items():
    df.write_parquet(DATA_DIR / f"trips-{date}.parquet")

In [ ]:
!ls -lh sample_data/citibike/trips-*.parquet

#### Verify

Let's check what we wrote. To count the rows on disk we use `scan_parquet` and compute `pl.len()`, which reads only the file metadata rather than loading 2.6 million rows just to count them. The glob pattern `trips-*.parquet` matches every daily file, so Polars treats them as one table.

The `assert` fails loudly if the number of rows on disk differs from the number of rows in memory.

In [ ]:
parquet_files = sorted(DATA_DIR.glob("trips-*.parquet"))
print(f"{len(parquet_files)} files, from {parquet_files[0].name} to {parquet_files[-1].name}")

n_rows_on_disk = (
    pl.scan_parquet(str(DATA_DIR / "trips-*.parquet")).select(pl.len()).collect().item()
)
assert n_rows_on_disk == trips_enriched.height
n_rows_on_disk

### Bonus: becoming faster by being lazy

So far we ran the cleaning step by step so you could inspect every stage. That's the *eager* style: each line executes immediately.

With the **lazy API** you instead describe the *whole* computation first and let Polars optimize it before running anything. Because we defined our building blocks once (`scan_trips`, `in_march`, `is_not_a_trip`, `distance_km`), the lazy version is simply those same pieces chained together, **without any `.collect()` in between**.

`explain()` prints the optimized **query plan**. Read it from the bottom up. Look for `PROJECT` lines such as "PROJECT 10/13 COLUMNS": that is Polars telling you it only reads the columns it needs.

In [ ]:
trips_clean_lazy = (
    scan_trips()
    .drop_nulls(subset=REQUIRED_COLUMNS)
    .filter(in_march)
    .filter(~is_not_a_trip)
    .with_columns(distance=distance_km)
)


In [ ]:
print(trips_clean_lazy.explain())

Now let's time the two styles. `%%time` is a Jupyter "cell magic" that reports how long a cell takes. Run each cell a couple of times; the first run is slower because the files must be read from disk.

Don't expect a dramatic difference here: this is a single month of data, and both versions already use `scan_csv`. The benefit of lazy evaluation grows with the size and complexity of the query. Chapter 5 covers the lazy API in detail, including how to process data that does not fit in memory.

In [ ]:
%%time
trips_clean_eager_timed = (
    scan_trips()
    .collect()
    .drop_nulls(subset=REQUIRED_COLUMNS)
    .filter(in_march)
    .filter(~is_not_a_trip)
    .with_columns(distance=distance_km)
)

In [ ]:
%%time
trips_clean_collected = trips_clean_lazy.collect()

#### Prove the two pipelines agree

A faster pipeline is worthless if it gives different answers. We run the lazy result through the same enrichment functions and compare it with `trips_enriched` using `assert_frame_equal` from `polars.testing`.

We pass `check_row_order=False` because trips with identical timestamps may appear in a different order. If the two DataFrames differ in any way (values, columns, or data types), this raises an `AssertionError` that explains how.

In [ ]:
stations_lazy = compute_stations(trips_clean_collected)
station_neighborhoods_lazy = locate_stations(stations_lazy, neighborhoods).drop("n_matches")
trips_enriched_lazy = add_boroughs_and_neighborhoods(
    trips_clean_collected, station_neighborhoods_lazy
)

assert_frame_equal(trips_enriched, trips_enriched_lazy, check_row_order=False)
print("The eager and lazy pipelines produce identical results.")

## Takeaways

In this notebook you saw that:

- **`scan_*` + `collect()`** (lazy) lets Polars read only the columns it needs; `read_*` (eager) loads everything immediately.
- **Expressions** are reusable descriptions of computations. Define a rule once and use it in any pipeline.
- **DataFrames are immutable**, so you can keep every intermediate stage under its own name and re-run cells safely.
- **Enums** make categorical columns compact and strict.
- **`validate=` on joins** documents the relationship you expect and fails loudly when the data disagrees.
- **Always log what you remove** (row counts per cleaning step) and **check your assumptions** with assertions and `polars.testing`.
- **Parquet** keeps your schema and is much faster than CSV to read back.
- Polars does tables; use specialized libraries (here Shapely) for what Polars doesn't do, but pass whole columns, not row by row.

### What changed compared with the book

| Book | This notebook | Why |
|---|---|---|
| Custom Rust plugin (`polars_geo`) for distance and point-in-polygon | Polars expressions for distance; Shapely `STRtree` for point-in-polygon | Runs on Colab without compiling Rust; much faster than a cross join |
| `from plotnine import *` | `import plotnine as p9` | Explicit is better than implicit |
| `pl.Categorical` | `pl.Enum` | Categories are known in advance |
| `drop_nulls()` on all columns | `drop_nulls(subset=...)` | Be explicit about which missing values matter |
| Eager and lazy versions differ slightly | Shared building blocks + `assert_frame_equal` | Guarantees the versions agree |
| Joins without checks | `validate=` | Fail loudly on unexpected data |

### Next up

In Chapter 2 we look at installing Polars, configuring it, and getting started on your own projects.

*Tip for instructors:* to catch API changes when you upgrade Polars, run the notebook with Polars deprecation warnings turned into errors (for example `warnings.filterwarnings("error", category=pl.exceptions.DeprecationWarning)` before the pipeline cells) and see what breaks.

### Try it yourself


1. Add a column `speed_kmh` (distance divided by duration in hours). Which borough has the highest median speed?

2. Compute the number of trips per `rider_type` and `bike_type`. Hint: `group_by(...).agg(pl.len())`.

3. Change `every="1d"` to `every="1h"` in `group_by_dynamic` and plot the last week of March.

4. Which neighborhood has the most trips starting in it? Use `group_by("neighborhood_start")` and `sort`.